## How to represent text for AI


In [1]:
import numpy as np

def one_hot_encoding(sentence):
    words = sentence.lower().split()
    vocab = sorted(set(words))
    words_to_index = {word: idx for idx, word in enumerate(vocab)}
    one_hot_matrix = np.zeros((len(words), len(vocab)), dtype=int)
    for i, word in enumerate(words):
        one_hot_matrix[i, words_to_index[word]] = 1

    return one_hot_matrix, vocab

sentence = "I want to go to Berkeley this summer"
one_hot_matrix, voc = one_hot_encoding(sentence)
print("Vocabulary:", voc)
print("One-hot encoded matrix:\n", one_hot_matrix)

Vocabulary: ['berkeley', 'go', 'i', 'summer', 'this', 'to', 'want']
One-hot encoded matrix:
 [[0 0 1 0 0 0 0]
 [0 0 0 0 0 0 1]
 [0 0 0 0 0 1 0]
 [0 1 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [1 0 0 0 0 0 0]
 [0 0 0 0 1 0 0]
 [0 0 0 1 0 0 0]]


In [2]:
def bag_of_words(sentences):
    tokenized_sentences = [sentence.lower().split() for sentence in sentences]
    flat_words = [word for sentence in tokenized_sentences for word in sentence]
    vocab = sorted(set(flat_words))
    words_to_index = {word: idx for idx, word in enumerate(vocab)}
    bow_matrix = np.zeros((len(sentences), len(vocab)), dtype=int)
    for i, sentence in enumerate(tokenized_sentences):
        for word in sentence:
            if word in words_to_index:
                bow_matrix[i, words_to_index[word]] += 1
    return bow_matrix, vocab

corpus = ["I'm not confident about my knowledge of machine learning.",
          "I want to learn more about deep learning and neural networks.",
            "I have some experience with data science but want to improve."]

bow_matrix, vocab = bag_of_words(corpus)
print("Vocabulary:", vocab)
print("Bag-of-Words matrix:\n", bow_matrix)

Vocabulary: ['about', 'and', 'but', 'confident', 'data', 'deep', 'experience', 'have', 'i', "i'm", 'improve.', 'knowledge', 'learn', 'learning', 'learning.', 'machine', 'more', 'my', 'networks.', 'neural', 'not', 'of', 'science', 'some', 'to', 'want', 'with']
Bag-of-Words matrix:
 [[1 0 0 1 0 0 0 0 0 1 0 1 0 0 1 1 0 1 0 0 1 1 0 0 0 0 0]
 [1 1 0 0 0 1 0 0 1 0 0 0 1 1 0 0 1 0 1 1 0 0 0 0 1 1 0]
 [0 0 1 0 1 0 1 1 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1]]


In [3]:
def compute_tf(sentences):
    vocab = sorted(set(word for sentence in sentences for word in sentence.lower().split()))
    words_to_index = {word: idx for idx, word in enumerate(vocab)}
    tf_matrix = np.zeros((len(sentences), len(vocab)), dtype=np.float32)
    for i, sentence in enumerate(sentences):
        words = sentence.lower().split()
        word_count = len(words)
        for word in words:
            if word in words_to_index:
                tf_matrix[i, words_to_index[word]] += 1 / word_count

    return tf_matrix, vocab

def compute_idf(sentences, voc):
    num_docs = len(sentences)
    idf_vector = np.zeros(len(voc), dtype=np.float32)
    words_index = {word: idx for idx, word in enumerate(voc)}
    for word in voc:
        df = sum(1 for sentence in sentences if word in sentence.lower().split())
        idf_vector[words_index[word]] = np.log(num_docs / (df + 1)) + 1

    return idf_vector

def compute_tfidf(sentences):
    tf_matrix, vocab = compute_tf(sentences)
    idf_vector = compute_idf(sentences, vocab)
    tfidf_matrix = tf_matrix * idf_vector
    return tfidf_matrix, vocab

tfidf_matrix, vocab = compute_tfidf(corpus)
print("Vocabulary:", vocab)
print("TF-IDF matrix:\n", tfidf_matrix)
      

Vocabulary: ['about', 'and', 'but', 'confident', 'data', 'deep', 'experience', 'have', 'i', "i'm", 'improve.', 'knowledge', 'learn', 'learning', 'learning.', 'machine', 'more', 'my', 'networks.', 'neural', 'not', 'of', 'science', 'some', 'to', 'want', 'with']
TF-IDF matrix:
 [[0.11111111 0.         0.         0.1561628  0.         0.
  0.         0.         0.         0.1561628  0.         0.1561628
  0.         0.         0.1561628  0.1561628  0.         0.1561628
  0.         0.         0.1561628  0.1561628  0.         0.
  0.         0.         0.        ]
 [0.09090909 0.12776956 0.         0.         0.         0.12776956
  0.         0.         0.09090909 0.         0.         0.
  0.12776956 0.12776956 0.         0.         0.12776956 0.
  0.12776956 0.12776956 0.         0.         0.         0.
  0.09090909 0.09090909 0.        ]
 [0.         0.         0.12776956 0.         0.12776956 0.
  0.12776956 0.12776956 0.09090909 0.         0.12776956 0.
  0.         0.         0.    

## Embedding, application and representation

In [6]:
!pip install gensim
!pip install adjustText
!pip install umap-learn

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   - -------------------------------------- 1.0/24.4 MB 25.4 MB/s eta 0:00:01
   ------ --------------------------------- 4.2/24.4 MB 12.6 MB/s eta 0:00:02
   ------------- -------------------------- 8.4/24.4 MB 17.3 MB/s eta 0:00:01
   ----------------- ---------------------- 10.5/24.4 MB 15.2 MB/s eta 0:00:01
   --------------------- ------------------ 13.1/24.4 MB 13.3 MB/s eta 0:00:01
   -------------------------- ------------- 16.0/24.4 MB 13.4 MB/s eta 0:00:01
   -------------------------------- ------- 19.9/24.4 MB 14.8 MB/s eta 0:00:01
   ------------------------------------ --- 22.3/24.4 MB 14.0 MB/s eta 0:00:01
   ---------------------------------------  24.4/24.4 MB 15.0 MB/s eta 0:00:01
   ---------------------------------------- 24.4/24.4 MB 12.9 MB/s  0:00:01


   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   ----------------------------- ---------- 9.7/13.1 MB 46.5 MB/s eta 0:00:01
   ---------------------------------------  12.8/13.1 MB 47.3 MB/s eta 0:00:01
   ---------------------------------------- 13.1/13.1 MB 28.3 MB/s  0:00:00

  Attempting uninstall: numpy

    Found existing installation: numpy 2.4.3

   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
    Uninstalling numpy-2.4.3:
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
      Successfully uninstalled numpy-2.4.3
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------- 0/3 [numpy]
   ---------------------------------------

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.3.0 requires dill<0.4.1,>=0.3.0, but you have dill 0.4.1 which is incompatible.
datasets 4.3.0 requires fsspec[http]<=2025.9.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
datasets 4.3.0 requires multiprocess<0.70.17, but you have multiprocess 0.70.19 which is incompatible.
llama-index-legacy 0.9.48.post4 requires tenacity<9.0.0,>=8.2.0, but you have tenacity 9.1.4 which is incompatible.
llama-index-multi-modal-llms-openai 0.1.9 requires llama-index-core<0.11.0,>=0.10.1, but you have llama-index-core 0.14.18 which is incompatible.
llama-index-multi-modal-llms-openai 0.1.9 requires llama-index-llms-openai<0.2.0,>=0.1.1, but you have llama-index-llms-openai 0.6.26 which is incompatible.
llama-index-program-openai 

In [16]:
!pip install --upgrade numpy==2.3.0 numba umap-learn

Found existing installation: numba 0.65.0
Uninstalling numba-0.65.0:
  Successfully uninstalled numba-0.65.0
Found existing installation: llvmlite 0.47.0
Uninstalling llvmlite-0.47.0:
  Successfully uninstalled llvmlite-0.47.0


You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.


Found existing installation: numba 0.65.0
Uninstalling numba-0.65.0:
  Successfully uninstalled numba-0.65.0
Found existing installation: llvmlite 0.47.0
Uninstalling llvmlite-0.47.0:
  Successfully uninstalled llvmlite-0.47.0


You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.


   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ------ --------------------------------- 2.1/12.9 MB 11.8 MB/s eta 0:00:01
   ------------- -------------------------- 4.2/12.9 MB 11.5 MB/s eta 0:00:01
   ------------------- -------------------- 6.3/12.9 MB 10.4 MB/s eta 0:00:01
   -------------------------- ------------- 8.4/12.9 MB 10.0 MB/s eta 0:00:01
   ----------------------------- ---------- 9.4/12.9 MB 10.0 MB/s eta 0:00:01
   ------------------------------------ --- 11.8/12.9 MB 9.5 MB/s eta 0:00:01
   ---------------------------------------- 12.9/12.9 MB 9.3 MB/s  0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.0
    Uninstalling numpy-2.3.0:
      Successfully uninstalled numpy-2.3.0


...

   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ------ --------------------------------- 2.1/12.9 MB 11.8 MB/s eta 0:00:01
   ------------- -------------------------- 4.2/12.9 MB 11.5 MB/s eta 0:00:01
   ------------------- -------------------- 6.3/12.9 MB 10.4 MB/s eta 0:00:01
   -------------------------- ------------- 8.4/12.9 MB 10.0 MB/s eta 0:00:01
   ----------------------------- ---------- 9.4/12.9 MB 10.0 MB/s eta 0:00:01
   ------------------------------------ --- 11.8/12.9 MB 9.5 MB/s eta 0:00:01
   ---------------------------------------- 12.9/12.9 MB 9.3 MB/s  0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.0
    Uninstalling numpy-2.3.0:
      Successfully uninstalled numpy-2.3.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
librosa 0.11.0 requires numba>=0.51.0, which is not installed.
umap-learn 0.5.12 requires numba>=0.51.2, which is not installed.
datasets 4.3.0 requires dill<0.4.1,>=0.3.0, but you have dill 0.4.1 which is incompatible.
datasets 4.3.0 requires fsspec[http]<=2025.9.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
datasets 4.3.0 requires multiprocess<0.70.17, but you have multiprocess 0.70.19 which is incompatible.
llama-index-legacy 0.9.48.post4 requires tenacity<9.0.0,>=8.2.0, but you have tenacity 9.1.4 which is incompatible.
llama-index-multi-modal-llms-openai 0.1.9 requires llama-index-core<0.11.0,>=0.10.1, but you have llama-index-core 0.14.18 which is incompatible.
llama-index-multi-modal-llms-openai 0.1.9 requires llama-index-llms-openai<0.2.0,>=0.1.1, but you have llama-index-llms-openai 0.

INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.

The conflict is caused by:
    The user requested llvmlite==0.42.0
    numba 0.60.0 depends on llvmlite<0.44 and >=0.43.0dev0

Additionally, some packages in these conflicts have no matching distributions available for your environment:
    llvmlite

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict



ERROR: Cannot install llvmlite==0.42.0 and numba==0.60.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


  Using cached numba-0.65.0-cp311-cp311-win_amd64.whl.metadata (2.8 kB)
  Using cached llvmlite-0.47.0-cp311-cp311-win_amd64.whl.metadata (4.9 kB)
Using cached numba-0.65.0-cp311-cp311-win_amd64.whl (2.7 MB)
Using cached llvmlite-0.47.0-cp311-cp311-win_amd64.whl (38.1 MB)

   ---------------------------------------- 0/2 [llvmlite]
   ---------------------------------------- 0/2 [llvmlite]
   ---------------------------------------- 0/2 [llvmlite]
   ---------------------------------------- 0/2 [llvmlite]
   ---------------------------------------- 0/2 [llvmlite]
   ---------------------------------------- 0/2 [llvmlite]
   ---------------------------------------- 0/2 [llvmlite]
   ---------------------------------------- 0/2 [llvmlite]
   ---------------------------------------- 0/2 [llvmlite]
   ---------------------------------------- 0/2 [llvmlite]
   ---------------------------------------- 0/2 [llvmlite]
   -------------------- ------------------- 1/2 [numba]
   ------------------

In [17]:
import numpy as np
import pandas as pd
import os
import re
import time
import nltk
from gensim.models import Word2Vec
from tqdm import tqdm
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from adjustText import adjust_text 
from umap import UMAP


ImportError: DLL load failed while importing _dynfunc: An Application Control policy has blocked this file.

In [19]:

import urllib.request
import zipfile
import os

# Download IMDB dataset
url = "https://github.com/SalvatoreRa/tutorial/blob/main/datasets/IMDB.zip?raw=true"
zip_path = "IMDB.zip"

print("Downloading IMDB dataset...")
urllib.request.urlretrieve(url, zip_path)
print("Download complete!")

# Extract the zip file
print("Extracting files...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")
print("Extraction complete!")

# List extracted files
extracted_files = os.listdir(".")
print("Extracted files:", [f for f in extracted_files if f.endswith('.csv')])

df = pd.read_csv("IMDB Dataset.csv")
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


Download complete!
Extracting files...
Extraction complete!
Extracted files: ['IMDB Dataset.csv']
Dataset loaded: 50000 rows, 2 columns


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [22]:
def preprocessing_reviews(reviews):

    """
    simple preprocessing: splitting on the space and remove word less than 1 chr
    """

    processed_reviews = []

    for review in tqdm(reviews):
        review = re.sub('<[^>]+>', '', review)
        processed = re.sub('[^a-zA-Z ]', '', review)
        words = processed.split()
        processed_reviews.append(' '.join([word.lower() for word in words if len(word) > 1]))
    return processed_reviews

df['reviews_processed'] = preprocessing_reviews(df['review'])
df['tokens'] = df['reviews_processed'].apply(nltk.word_tokenize)
df.head()

100%|██████████| 50000/50000 [00:02<00:00, 19780.33it/s]


,review,sentiment,reviews_processed,tokens
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,"[one, of, the, other, reviewers, has, mentione..."
1,A wonderful little production. <br /><br />The...,positive,wonderful little production the filming techni...,"[wonderful, little, production, the, filming, ..."
2,I thought this was a wonderful way to spend ti...,positive,thought this was wonderful way to spend time o...,"[thought, this, was, wonderful, way, to, spend..."
3,Basically there's a family where a little boy ...,negative,basically theres family where little boy jake ...,"[basically, theres, family, where, little, boy..."
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love in the time of money is vi...,"[petter, matteis, love, in, the, time, of, mon..."


In [21]:

# Download required NLTK data
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
print("NLTK data downloaded successfully!")


NLTK data downloaded successfully!


In [ ]:
start_time = time.time()
# embedding
model = Word2Vec(sentences=df['tokens'].tolist(),
                 sg=1,
                 vector_size=100,
                 window=5,
                 workers=4)

print(f'Time needed : {(time.time() - start_time) / 60:.2f} mins')

In [ ]:
# Entire set of words in the model
all_words = list(model.wv.index_to_key)  
all_vectors = np.array([model.wv[word] for word in all_words])

# Highlighted words and their vectors
highlight_words = ['Berlin', 'Paris', 'London','Rome', 'Italy',
                   'France', 'Germany', 'England', 'movie', 'production', 'good', 'bad']
highs = [w.lower() for w in highlight_words]
indices = [all_words.index(word) for word in highs if word in all_words]
highlight_vectors = np.array([all_vectors[index] for index in indices])

linked = linkage(highlight_vectors, 'ward')

plt.figure(figsize=(5, 4))
dendrogram(linked,
           orientation='top',
           labels=highlight_words,
           distance_sort='descending',
           show_leaf_counts=True)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Words')
plt.ylabel('Euclidean distances')
plt.xticks(rotation=45)
plt.savefig('word_dendrogram.jpg', format='jpeg', bbox_inches='tight')
plt.show()

In [ ]:
# Apply t-SNE to the entire set of vectors
tsne = TSNE(n_components=2, random_state=0)
Y_tsne = tsne.fit_transform(all_vectors)

highlight_words = ['berlin', 'rome', 'London', 'France', 'Germany',
                    'movie', 'production', 'mother', 'family']

highs = [w.lower() for w in highlight_words]
indices = [all_words.index(word) for word in highs if word in all_words]
highlight_vectors = np.array([all_vectors[index] for index in indices])
Y_highlight = Y_tsne[indices]


plt.figure(figsize=(10, 7))


sns.scatterplot(x=Y_tsne[:, 0], y=Y_tsne[:, 1], color="lightgrey", alpha=0.3)

# Plot highlighted words 
palette = sns.color_palette("hsv", len(highlight_words))
texts = []
for i, word in enumerate(highlight_words):
    plt.scatter(Y_highlight[i, 0], Y_highlight[i, 1], color=palette[i], s=100, label=word)
    # adjust text
    texts.append(plt.text(Y_highlight[i, 0], Y_highlight[i, 1], word, fontsize=12))

adjust_text(texts, arrowprops=dict(arrowstyle='->', color='red'))

plt.title('t-SNE visualization of Word2Vec embeddings', fontsize=20)
plt.xlabel('Component 1')
plt.ylabel('Component 2')


plt.grid(True)
plt.legend(title='Highlighted Words', title_fontsize='13', fontsize='11')
plt.savefig('word_tsne.jpg', format='jpeg')
plt.show()

In [ ]:
# Apply UMAP to the entire set of vectors
umap = UMAP(n_components=2, random_state=42)  
Y_umap = umap.fit_transform(all_vectors)

Y_highlight = Y_umap[indices]


plt.figure(figsize=(10, 7))
sns.scatterplot(x=Y_umap[:, 0], y=Y_umap[:, 1], color="lightgrey", alpha=0.3)

palette = sns.color_palette("hsv", len(highlight_words))
texts = []
for i, word in enumerate(highlight_words):
    plt.scatter(Y_highlight[i, 0], Y_highlight[i, 1], color=palette[i], s=100, label=word)

    texts.append(plt.text(Y_highlight[i, 0], Y_highlight[i, 1], word, fontsize=12))

adjust_text(texts, arrowprops=dict(arrowstyle='->', color='red'))

plt.title('UMAP visualization of Word2Vec embeddings', fontsize=20)
plt.xlabel('Component 1')
plt.ylabel('Component 2')


plt.grid(True)
plt.legend(title='Highlighted Words', title_fontsize='13', fontsize='11')
plt.savefig('word_umap.jpg', format='jpeg')
plt.show()

In [ ]:
def plot_vectors_and_angle(v1, v2):
    
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    cosine_similarity = dot_product / (norm_v1 * norm_v2)
    angle_radians = np.arccos(cosine_similarity)
    angle_degrees = np.degrees(angle_radians)

   
    fig, ax = plt.subplots(figsize=(5, 5))
    
  
    ax.quiver(0, 0, v1[0], v1[1], angles='xy', scale_units='xy', scale=1, color='r', label=f"Vector 1: {v1}")
    ax.quiver(0, 0, v2[0], v2[1], angles='xy', scale_units='xy', scale=1, color='b', label=f"Vector 2: {v2}")

   
    start_angle = np.arctan2(v1[1], v1[0])
    if np.cross(v1, v2) < 0:
        angle_radians = -angle_radians

  
    theta = np.linspace(start_angle, start_angle + angle_radians, 100)
    r = 0.5 * min(np.linalg.norm(v1), np.linalg.norm(v2))  
    x = r * np.cos(theta)
    y = r * np.sin(theta)

   
    ax.plot(x, y, linestyle='-', color='green', lw=2)

    
    midpoint = (start_angle + angle_radians / 2)
    ax.annotate(r'$\theta$', xy=(r * np.cos(midpoint), r * np.sin(midpoint)), xytext=(20, 10), 
                textcoords='offset points', fontsize=16, arrowprops=dict(arrowstyle='->', lw=0.5))

    
    max_range = np.max(np.abs(np.vstack([v1, v2, [x.max(), y.max()]]))) * 1.1  # 10% padding
    ax.set_xlim([-max_range, max_range])
    ax.set_ylim([-max_range, max_range])

   
    plt.grid(True)
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.title(f'Angle between vectors: {angle_degrees:.2f} degrees')
    plt.suptitle(f'Similarity between vectors: {cosine_similarity:.2f}', fontsize=10, y=.95)
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.legend(loc='lower right')
    plt.savefig('cosine_similarity.jpg', format='jpeg', bbox_inches='tight')
    plt.show()

    return cosine_similarity, angle_degrees

# Example usage
v1 = np.array([2, 3])
v2 = np.array([-1, 2])
cos_sim, angle = plot_vectors_and_angle(v1, v2)

In [ ]:

word_1 = "good"
syn = "great"
ant = "bad"
most_sim =model.wv.most_similar("good")
print("Top 3 most simalr words to {} are :{}".format(word_1, most_sim[:3]))

synonyms_dist = model.wv.distance(word_1, syn)
antonyms_dist = model.wv.distance(word_1, ant)
print("Synonyms {}, {} have cosine distance: {}".format(word_1, syn, synonyms_dist))
print("Antonyms {}, {} have cosine distance: {}".format(word_1, ant, antonyms_dist))
a = 'king'
a_star = 'man'
b = 'woman'
b_star= model.wv.most_similar(positive=[a, b], negative=[a_star])
print("{} is to {} as {} is to: {} ".format(a, a_star, b, b_star[0][0]))